# 40. The probe along the stream, not just at its ends

Notebook 38 asked what the direct probe says at the start of a stream and at
its end. It answered: accuracy does not fall, it rises. But two points cannot
show the shape of what happened in between. A probe that dipped in the middle
and came back would look exactly the same from the ends.

That matters because the indicators do most of their moving early. On seed 2
effective rank loses 38 percent of its value after a single task. If the model
really were losing the ability to learn, that is where the probe should sag.

This notebook probes five points on the stream: 0, 2, 4, 6 and 8 tasks. It uses
the models the run kept for exactly this, at boundaries 2, 4 and 6.

**Only seeds 1 and 2.** Seed 0 ran before the code that keeps middle models
existed, so it has no files at 2, 4 and 6 and stays at two points.

**All five points run in one session per seed.** Repeats of this probe inside a
session differ by about 3 points of accuracy; repeats across sessions have
differed by as much as 8.4. A curve whose points came from different sessions
would mix the session with the position in the stream. So the two endpoints are
measured again here rather than copied from notebook 38's results, and the
notebook works through one seed completely before starting the next. If you run
out of session, finishing a seed is the natural place to stop.

**Nothing trains a stream.** Every model already exists. This loads them and
runs the same fork-and-discard probe notebook 38 ran, with every protocol
setting unchanged: Tiny ImageNet task 2, 4,800 training images, three learning
rates, 250 steps, eleven repetitions, rate chosen on validation and scored on
test.

**Cost.** 330 probe runs at roughly 0.6 minutes each, so about 3.3 hours of GPU
time for a clean start, plus about 3 GB of model downloads. Half of that is one
seed. Results are saved to R2 as they finish and the notebook resumes, so a
session that ends early loses nothing.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.probe import noise_floor, probe_once, snapshot, unchanged

import datetime
import torch
from tqdm.auto import tqdm

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)

## 2. Settings

The same protocol as notebook 38. Only the seeds, the probe points and the
saving label change.

In [ ]:
STUDY_ID = "indicator_stream_v1"
BUCKET = "uncle-experiments"
ORDER = "order_01"

# Seeds 1 and 2 only. Seed 0's stream ran before the code that keeps a model at
# the middle boundaries, so it has no files at 2, 4 and 6. Including it would
# give a two-point seed sitting in a table of five-point seeds, which invites
# exactly the comparison this notebook exists to avoid.
SEEDS = [1, 2]

# Where on the stream to probe. 0 is the fresh generator, which the seed
# rebuilds exactly and needs no file. 8 is the finished model. 2, 4 and 6 are
# the kept boundary models.
#
# 0 and 8 are measured again here rather than taken from notebook 38's results.
# They are the same models, but the probe's spread across sessions is wider than
# its spread inside one, so borrowing them would put a session difference in the
# middle of the curve and there would be no way to tell it from a real dip.
POINTS = [0, 2, 4, 6, 8]
FINAL_POINT = 8               # the only point whose model is the job's own save

# A separate file from notebook 38's. The protocol is identical, so the numbers
# can sit in one table afterwards, but writing to its file would mean every
# session of this run rewrites 198 finished probe runs, and a bad write would
# lose them.
RUN_LABEL = "curve1"
RESUME_KEY = f"uncle/matched_probe/{STUDY_ID}/{ORDER}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 10               # upload after this many new runs

CLASSES_HINT = 10             # classes per task, confirmed against the plan in section 4
EUROSAT = "eurosat"

# Off, as in notebook 38. EuroSAT needs about forty repetitions before its
# noise floor is narrow enough to show anything, and at five points that is a
# run of its own.
INCLUDE_EUROSAT = False

# The Tiny ImageNet probe task, unchanged from notebooks 35 and 38.
REFERENCE_TASKS = ["2"]

# Unchanged from notebook 38. Changing any of these would make the curve
# incomparable with the endpoints already measured, which is the one thing this
# notebook must not do.
#
# The record shows the selected rate was always 1e-4 or 3e-4 and never 1e-3.
# Dropping 1e-3 would cut the run by a third. It is kept because a model in the
# middle of a stream has not been probed before and may want a rate the
# endpoints did not.
RATES = [1e-4, 3e-4, 1e-3]
STEPS = 250                   # never varied between probe points
BATCH_SIZE = 64
FEATURE_IMAGES = 2000         # for the effective rank

# Equal at every point, so the noise floor is the same shape everywhere.
REPETITIONS_AT_ZERO = 11
REPETITIONS_LATER = 11

ARMS = [
    ("full", False, True),        # generator and code both train
]

TRAIN_PER_CLASS = 480
VAL_PER_CLASS = 20
TEST_PER_CLASS = 200
SPLIT_SEED = 7                # fixes which images land in which split

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
EUROSAT_ROOT = Path("/content/data") if IN_COLAB else REPO / "data"

PROBE_TASKS = ([EUROSAT] if INCLUDE_EUROSAT else []) + REFERENCE_TASKS
if not PROBE_TASKS:
    raise ValueError("No probe tasks. Turn EuroSAT on or name a reference task.")
if 0 not in POINTS:
    raise ValueError("Point 0 is the baseline every other point is measured against.")
TOTAL_RUNS = (len(PROBE_TASKS) * len(RATES) * len(ARMS) * len(POINTS)
              * REPETITIONS_LATER * len(SEEDS))

# Recorded with every run. The noise floor is built from the t = 0 repetitions,
# so a point measured on a different machine from the baseline could show a
# difference between the machines as a result. SESSION does the same job for
# time: two runs sharing it ran in one sitting.
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", GPU_NAME)
print("Session stamp:", SESSION)
print(f"Probe runs: {TOTAL_RUNS}  ({len(PROBE_TASKS)} task(s), {len(RATES)} rates, "
      f"{len(ARMS)} arm, {len(POINTS)} points, {REPETITIONS_LATER} repetitions, "
      f"{len(SEEDS)} seeds)")
print(f"At roughly 0.6 minutes each that is about {TOTAL_RUNS * 0.6 / 60:.1f} hours "
      f"for a clean start, or about {TOTAL_RUNS * 0.6 / 60 / len(SEEDS):.1f} hours per seed. "
      "Runs already saved are skipped, and the grid cell below reports how many "
      "are actually new.")
print("Probe points:", POINTS)
print("Rate chosen on validation accuracy, result read from the test split.")
print(f"Training images per probe task: {CLASSES_HINT * TRAIN_PER_CLASS}, "
      f"validation {CLASSES_HINT * VAL_PER_CLASS}. Saving under: {RUN_LABEL}")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Build the images and find every model

Five probe points need four model files per seed: the kept models at 2, 4 and 6
and the job's own finished save at 8. Point 0 needs no file, because the seed
rebuilds the fresh generator exactly.

Every file is checked here, before a single probe runs. A missing model found
three hours in would waste the session.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]

if FINAL_POINT != len(ORDER_TASKS):
    raise ValueError(f"FINAL_POINT is {FINAL_POINT} but the order has "
                     f"{len(ORDER_TASKS)} tasks.")

overlap = set(REFERENCE_TASKS) & set(ORDER_TASKS)
if overlap:
    raise ValueError(f"These probe tasks are in the learning order: {sorted(overlap)}")

needed = tuple(ORDER_TASKS + REFERENCE_TASKS)
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")


def job_for(seed):
    return next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{seed}")


def model_key(seed, t):
    """Where that seed's model after t tasks lives in R2.

    The finished model is wherever the run's own progress file points, which is
    a rolling slot the run rewrites. The middle models were copied out of that
    slot to fixed names the run never touches, by notebook 36.
    """
    job = job_for(seed)
    if t == FINAL_POINT:
        progress = storage.progress(job)
        if progress is None or progress["status"] != "complete":
            raise RuntimeError(f"Seed {seed}'s stream is not finished in R2.")
        # Checks size and sha256 from the object's metadata, downloading nothing.
        storage.validate_artifact(progress["checkpoint"])
        return progress["checkpoint"]["key"]
    return f"{storage.prefix}/{job['id']}/boundary_{t}.pt"


# Every file, before anything runs. store.info reads the object's metadata, so
# this costs a few seconds and no bandwidth.
print("Checking every model this run needs:")
KEYS = {}
for seed in SEEDS:
    for t in POINTS:
        if t == 0:
            print(f"  seed {seed}, t = 0: no file needed, the seed rebuilds it")
            continue
        key = model_key(seed, t)
        info = store.info(key)
        if info is None:
            raise RuntimeError(
                f"No model for seed {seed} at boundary {t}. Expected: {key}. "
                "Either that boundary was never kept, or the key is wrong. "
                "Take that point out of POINTS, or take the seed out of SEEDS.")
        KEYS[(seed, t)] = key
        print(f"  seed {seed}, t = {t}: {info['ContentLength'] / 1e6:>6.1f} MB  {key}")
print("Every model is present.")


def build(seed, t):
    """That seed's model after t tasks, ready to probe.

    t = 0 is the generator the seed builds before any training, so nothing is
    loaded. Every other point loads a saved file and checks that the file really
    is from where it claims: `job_requests_done` counts the learn requests the
    run had finished when it saved, and every request in this study is a learn.
    """
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": tuple(("learn", task) for task in ORDER_TASKS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if t == 0:
        return target, hypernet, trainer

    path = store.download(KEYS[(seed, t)], CACHE / "cache")
    state = torch.load(path, map_location="cpu", weights_only=False)
    done_here = state.get("job_requests_done")
    if done_here != t:
        raise RuntimeError(
            f"The file for seed {seed} boundary {t} says {done_here} tasks were "
            "finished when it was saved. Probing it would put a model from the "
            "wrong place in the stream on the curve.")
    if state.get("active_request") is not None:
        raise RuntimeError(
            f"The file for seed {seed} boundary {t} was saved part way through a "
            "task, so it is not a boundary model.")
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    del state
    return target, hypernet, trainer

In [ ]:
from torchvision.datasets import EuroSAT
from torchvision.transforms import ToTensor
from torch.utils.data import Subset

def balanced_split(labels, counts, seed):
    """Row numbers for several disjoint splits, equal numbers from every class.

    `labels` holds one label per row and `counts` says how many images per
    class each split gets. Working on the labels alone means no image is read
    here, so this stays fast on a dataset of 27,000 files.
    """
    by_class = {}
    for row, label in enumerate(labels):
        by_class.setdefault(int(label), []).append(row)
    if len(by_class) != CLASSES:
        raise RuntimeError(f"Expected {CLASSES} classes, found {len(by_class)}.")
    stream = torch.Generator().manual_seed(seed)
    splits = [[] for _ in counts]
    for label, rows in sorted(by_class.items()):
        if len(rows) < sum(counts):
            raise RuntimeError(f"Class {label} has {len(rows)} images, needs {sum(counts)}.")
        order = [rows[j] for j in torch.randperm(len(rows), generator=stream).tolist()]
        at = 0
        for which, how_many in enumerate(counts):
            splits[which] += order[at:at + how_many]
            at += how_many
    taken = [row for split in splits for row in split]
    assert len(set(taken)) == len(taken), "An image landed in two splits."
    return splits

assert CLASSES_HINT == CLASSES, f"Settings assume {CLASSES_HINT} classes, the plan says {CLASSES}"
probe_sets = {}

if INCLUDE_EUROSAT:
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(), download=True)

    # The generator's output layout is fixed when it is built, and the backbone
    # expects 64x64. Both have to match or the probe measures the wrong thing.
    assert len(eurosat.classes) == CLASSES, f"EuroSAT has {len(eurosat.classes)} classes, need {CLASSES}"
    assert eurosat[0][0].shape == (3, 64, 64), f"Unexpected image shape {eurosat[0][0].shape}"

    # `samples` is a list of (path, label), so this reads no images.
    parts = balanced_split([label for _, label in eurosat.samples],
                           (TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS), SPLIT_SEED)
    probe_sets[EUROSAT] = dict(zip(("train", "val", "test"),
                                   (Subset(eurosat, part) for part in parts)))
else:
    print("EuroSAT is off for this pass.")

# The Tiny ImageNet task's training and validation splits are cut out of its
# own training images, so no image is ever both trained on and used to choose a
# learning rate. Its test split is the one the rest of the project uses, and it
# was the same 500 images in every pass, so test-set noise cannot explain any
# difference between passes.
for task in REFERENCE_TASKS:
    base = tasks[task]["train"]
    train_rows, val_rows = balanced_split(base.targets,
                                          (TRAIN_PER_CLASS, VAL_PER_CLASS), SPLIT_SEED)
    probe_sets[task] = {"train": Subset(base, train_rows),
                        "val": Subset(base, val_rows),
                        "test": tasks[task]["test"]}

for name, split in probe_sets.items():
    print(f"{name:<10} train {len(split['train']):>5}  val {len(split['val']):>5}  "
          f"test {len(split['test']):>5}")
print("Probe tasks:", PROBE_TASKS)

## 5. Run the grid

One seed at a time, and inside a seed one probe point at a time, lowest first.
Stopping after a finished seed leaves a complete curve for that seed, which is
the unit the comparison is made in.

In [ ]:
import time

# A run is identified by everything that makes it different from another run.
def identity(run):
    return (run["seed_label"], run["point"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])

def probe_seed(task, repetition):
    """Fixes the batch order and the fresh code. Depends on the task and the
    repetition, never on the probe point, so two points differ only by the
    model's state."""
    return 12345 + 1000 * repetition + (int(task) if task.isdigit() else 99)

PARTIAL = CACHE / "diagnostics" / f"probe_curve_{RUN_LABEL}_partial.jsonl"
PARTIAL.parent.mkdir(parents=True, exist_ok=True)

results = []
if store.info(RESUME_KEY) is not None:
    # One JSON object per line, so a half-written final line can be dropped
    # without losing everything before it.
    saved = store.download(RESUME_KEY, CACHE / "cache")
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}
new_runs = 0

def grid_at(seed, point, repetitions):
    """Every probe task, rate, arm and repetition at one probe point."""
    global new_runs
    todo = [(task, rate, arm, code_only, train_code, repetition)
            for task in PROBE_TASKS
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for repetition in range(repetitions)
            if (seed, point, task, rate, arm, repetition) not in done]
    if not todo:
        print(f"seed {seed}, t = {point}: already finished, skipping")
        return

    target, hypernet, trainer = build(seed, point)
    reference = snapshot(hypernet, trainer)
    bar = tqdm(todo, desc=f"seed {seed}  t={point}", unit="run")
    for task, rate, arm, code_only, train_code, repetition in bar:
        bar.set_postfix_str(f"{task} lr={rate:g} {arm} rep {repetition + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=task,
            train_set=probe_sets[task]["train"], test_set=probe_sets[task]["test"],
            validation_set=probe_sets[task]["val"],
            steps=STEPS, learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            seed=probe_seed(task, repetition), feature_images=FEATURE_IMAGES)
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix this "
                               "before trusting anything.")
        run.update(point=point, endpoint="E2", repetition=repetition,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60,
                   device=GPU_NAME, run_label=RUN_LABEL, session=SESSION,
                   train_images=len(probe_sets[task]["train"]))
        results.append(run)
        done.add(identity(run))
        # Append rather than rewrite, so saving stays cheap as the file grows.
        with PARTIAL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            store.upload(PARTIAL, RESUME_KEY, overwrite=True)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)

began = time.perf_counter()
for seed in SEEDS:
    for point in POINTS:
        grid_at(seed, point,
                repetitions=REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER)
store.upload(PARTIAL, RESUME_KEY, overwrite=True)

print(f"{len(results)} runs in total, {new_runs} in this session "
      f"({(time.perf_counter() - began) / 60:.1f} minutes).")
print("The model was verified unchanged after every one.")
missing = [(s, p) for s in SEEDS for p in POINTS
           if not any(r["seed_label"] == s and r["point"] == p for r in results)]
if missing:
    print("Still to do, as (seed, probe point):", missing)
    print("The session ran out of time. Run the notebook again from the top; "
          "everything already finished is skipped.")
else:
    print("Every seed has every probe point. Nothing left to run.")

# A curve whose points came from different sittings can show a session
# difference as a dip. This says, per seed, whether that happened.
for seed in SEEDS:
    sittings = {r.get("session") for r in results if r["seed_label"] == seed}
    sittings.discard(None)
    if len(sittings) > 1:
        print(f"Seed {seed} was measured across {len(sittings)} sessions. "
              "Treat differences between its points with care, and say so in "
              "the record.")

## 6. The result

One row per probe point. The noise floor is built from the spread of the eleven
repetitions at t = 0, so every point is judged against the same baseline the
endpoint run used.

A fall anywhere on this curve is plasticity loss. A flat or rising curve means
the model could learn an unseen task just as well in the middle of the stream
as at its start, which is what the indicators say should not be true.

In [ ]:
import statistics

def select(runs):
    """The run with the best validation accuracy. Ties go to the lower rate."""
    return min(runs, key=lambda r: (-r["val_accuracy"], r["learning_rate"]))

def per_repetition(seed, point, task, arm):
    """One number per repetition: the test accuracy of the selected run."""
    values = []
    for repetition in range(max(REPETITIONS_AT_ZERO, REPETITIONS_LATER)):
        runs = [r for r in results
                if r["seed_label"] == seed and r["point"] == point
                and r["task"] == task and r["arm"] == arm
                and r["repetition"] == repetition]
        if runs:
            values.append(select(runs)["test_accuracy"])
    return values

SEEDS_DONE = [s for s in SEEDS
              if all(per_repetition(s, p, PROBE_TASKS[0], "full") for p in POINTS)]
print("Seeds with every probe point finished:", SEEDS_DONE or "none yet")

verdicts = {}
for seed in SEEDS_DONE:
    for arm, _, _ in ARMS:
        for task in PROBE_TASKS:
            print("=" * 72)
            print(f"Seed {seed}, arm {arm}, probe task {task}")
            at = {p: per_repetition(seed, p, task, arm) for p in POINTS}
            spread = statistics.stdev(at[0]) if len(at[0]) > 1 else float("nan")
            base = statistics.mean(at[0])
            print(f"  spread of the {len(at[0])} repetitions at t = 0: {spread:.2f}")
            print(f"  {'t':>3} {'accuracy':>9} {'change':>8} {'floor':>7} "
                  f"{'ratio':>7}  verdict")
            for point in POINTS:
                here = statistics.mean(at[point])
                change = here - base
                floor = noise_floor(spread, spread, len(at[0]), len(at[point]))
                if point == 0:
                    verdict = "baseline"
                elif change > floor:
                    verdict = "higher than at t=0"
                elif change < -floor:
                    verdict = "LOWER, plasticity loss"
                else:
                    verdict = "unresolved, inside the floor"
                print(f"  {point:>3} {here:>9.2f} {change:>+8.2f} {floor:>7.2f} "
                      f"{here / base:>7.3f}  {verdict}")
                verdicts[(seed, arm, task, point)] = {
                    "accuracy": here, "change": change, "floor": floor,
                    "ratio": here / base, "spread_at_zero": spread,
                    "repetitions": len(at[point]), "verdict": verdict}

            # The shape, which is the question two points could not answer.
            middle = [verdicts[(seed, arm, task, p)]
                      for p in POINTS if p not in (0, FINAL_POINT)]
            dipped = [v for v in middle if v["change"] < -v["floor"]]
            if dipped:
                print(f"  Shape: the probe falls below its floor at {len(dipped)} "
                      f"of the {len(middle)} middle points. The endpoints alone "
                      "would have hidden this.")
            elif all(v["change"] > v["floor"] for v in middle):
                print("  Shape: every middle point is above the floor too, so the "
                      "rise is not an endpoint accident.")
            else:
                print("  Shape: no middle point falls below its floor. The probe "
                      "does not sag during the stream.")

## 7. Save

In [ ]:
import csv, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
seed_label = "seeds" + "".join(str(s) for s in SEEDS_DONE)
EXPORT = CACHE / "diagnostics" / f"probe_curve_{RUN_LABEL}_{ORDER}_{seed_label}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "40_probe_curve",
    "question": ("The probe does not fall between the start and the end of a "
                 "stream. Does it fall anywhere in between? The indicators do "
                 "most of their moving in the first task or two, so that is "
                 "where a real loss of plasticity should show."),
    "protocol": ("Fork and discard, identical to notebook 38. The model is "
                 "snapshotted, a fresh task code is trained on an unseen task, "
                 "the result is measured, and the model is restored and "
                 "verified unchanged. The learning rate is selected on a "
                 "validation split and the result read from a separate test "
                 "split. The only change from notebook 38 is that five probe "
                 "points are measured instead of two."),
    "limitations": [
        "Seeds 1 and 2 only. Seed 0's stream ran before the code that keeps a "
        "model at the middle boundaries, so it has no files at 2, 4 and 6.",
        "One class order and one probe task, Tiny ImageNet task 2.",
        "The middle models at 2, 4 and 6 were copied inside R2 from the run's "
        "own rolling save. Each file is checked here against the number of "
        "tasks it says were finished when it was saved, but there is no "
        "independent record of their contents.",
        "A point is compared against t = 0 of the same seed and the same "
        "session. Points from different sessions are not comparable at this "
        "precision, and the grid cell reports when a seed spans sessions.",
        "EuroSAT is off. Its spread needs about forty repetitions before a "
        "change of this size would clear its floor.",
        "Runs saved under different RUN_LABEL values used different settings "
        "and must not be pooled. The label is recorded with every run.",
    ],
    "commit": HEAD,
    "device": GPU_NAME,
    "session": SESSION,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "settings": {"run_label": RUN_LABEL, "include_eurosat": INCLUDE_EUROSAT,
                 "study": STUDY_ID, "order": ORDER, "seeds": SEEDS,
                 "points": POINTS,
                 "probe_tasks": PROBE_TASKS, "rates": RATES, "steps": STEPS,
                 "batch_size": BATCH_SIZE, "arms": [a[0] for a in ARMS],
                 "repetitions": [REPETITIONS_AT_ZERO, REPETITIONS_LATER],
                 "rate_selection": "highest validation accuracy, ties to the lower rate",
                 "split_seed": SPLIT_SEED,
                 "train_per_class": TRAIN_PER_CLASS, "val_per_class": VAL_PER_CLASS,
                 "test_per_class": TEST_PER_CLASS},
    "verdicts": {f"seed{seed}|{arm}|{task}|t{point}": value
                 for (seed, arm, task, point), value in verdicts.items()},
}
(EXPORT / "summary.json").write_text(json.dumps(record, indent=2), encoding="utf-8")

columns = ["run_label", "session", "seed_label", "point", "task", "arm", "repetition",
           "learning_rate", "train_images", "val_accuracy", "val_loss",
           "test_accuracy", "test_loss", "final_train_loss", "effective_rank",
           "minutes", "device"]
with (EXPORT / "runs.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

SAVE_VERIFIED = all((EXPORT / name).exists() and (EXPORT / name).stat().st_size > 0
                    for name in ("summary.json", "runs.csv"))
if SAVE_TO_R2 and SAVE_VERIFIED:
    for name in ("summary.json", "runs.csv"):
        store.upload(EXPORT / name, f"uncle/probe_curve/{STUDY_ID}/{EXPORT.name}/{name}",
                     overwrite=True)
SAVED_LOCATION = EXPORT
print("Saved to:", EXPORT)
print("Verified on disk:", SAVE_VERIFIED)

## 8. Release the GPU

Only once every file has been written and read back. If anything failed to save, the
runtime stays connected so the probe runs are not lost.

In [ ]:
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or empty.")
    print("Rerun section 7 before closing this runtime, or the probe runs are lost.")
elif missing:
    print("Results so far saved to:", SAVED_LOCATION)
    print("NOT releasing the GPU. These are not finished yet:", missing)
    print("Run the grid cell again to continue from where it stopped.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Everything saved. Leaving the runtime connected as asked.")
else:
    print("Everything saved. Releasing the GPU.")
    if IN_COLAB:
        from google.colab import runtime
        runtime.unassign()